# 03 · 宏观驱动因素

本 notebook 用**面板固定效应 + 聚类稳健标准误**估计宏观变量与评级变动的关系。

## 识别策略

$$\Delta \text{Rating}_{it} = \alpha_i + \gamma_t + \beta' X_{it} + \varepsilon_{it}$$

* $\alpha_i$：**国家固定效应**，吸收地理、制度传统等不随时间变化的特征
* $\gamma_t$：**时间固定效应**，吸收全球共同冲击（2008 金融危机、2020 疫情）
* 标准误按**国家聚类**：同一国家跨年的扰动项高度相关，忽略会严重低估标准误

## 因变量

| 因变量 | 类型 | 含义 |
| --- | --- | --- |
| `rating_change` | 连续 | 评级分值的一阶差分（+ 上调 / − 下调） |
| `is_downgrade` | 0/1 | 是否发生下调（线性概率模型） |
| `is_upgrade` | 0/1 | 是否发生上调（线性概率模型） |

> ⚠️ 这些系数应解释为**条件相关**，而非因果效应。评级机构使用前瞻性信息，
> 宏观变量与评级变动之间存在明显的反向因果与共同驱动因素（如政策预期）。

> ⚠️ 本 notebook 默认使用 `data/sample/` 下的**合成演示数据**；
> 数值不构成实证结论。接入真实数据后请替换 `PANEL_PATH`。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.analysis.panel_regression import (
    compare_results,
    fit_fixed_effects,
    fit_logit,
    fit_pooled_ols,
    run_driver_regressions,
)
from src.config import get_path
from src.features.macro_features import DEFAULT_FEATURES, FEATURE_GROUPS, add_derived_features
from src.visualization.plots import plot_coefficient_table, save_figure

PANEL_PATH = get_path("processed") / "panel_country_year.csv"
if PANEL_PATH.exists():
    panel = pd.read_csv(PANEL_PATH, encoding="utf-8-sig")
else:
    from src.clean.panel import build_country_year_panel
    from src.ingest.ratings import load_sample_ratings

    panel = build_country_year_panel(
        load_sample_ratings(),
        pd.read_csv(get_path("sample") / "macro_sample.csv", encoding="utf-8-sig"),
        start_year=2000,
        end_year=2023,
    )
panel = add_derived_features(panel, group_cols=("country_iso3", "agency"))

FEATURES = [c for c in DEFAULT_FEATURES if c in panel.columns]
print(f"可用解释变量 {len(FEATURES)} 个")
for group, names in FEATURE_GROUPS.items():
    print(f"  {group}: {[n for n in names if n in panel.columns]}")

## 1. 描述性统计与相关性

先看解释变量之间的相关性——高度共线的变量不宜同时进入模型。

In [ ]:
descriptive = panel[FEATURES].describe().T
descriptive["缺失率"] = panel[FEATURES].isna().mean().round(4)
descriptive.round(3)

In [ ]:
corr = panel[FEATURES].corr()
high = (
    corr.where(~np.eye(len(corr), dtype=bool))
    .stack()
    .sort_values(key=abs, ascending=False)
    .head(12)
)
print("相关性最高（绝对值）的变量对：")
display(high.to_frame("相关系数").round(3))

## 2. 主回归：双向固定效应 + 国家聚类

若环境中安装了 `linearmodels`，将使用 `PanelOLS`（统计推断更规范）；
否则自动退化为「虚拟变量 + 聚类稳健 OLS」，系数完全一致。

In [ ]:
results = run_driver_regressions(
    panel, FEATURES, targets=("rating_change", "is_downgrade", "is_upgrade")
)
for result in results.values():
    print(result.summary_text())
    print()

In [ ]:
coefficients = results["rating_change"].to_frame()
coefficients.sort_values("coef", key=abs, ascending=False).round(4)

In [ ]:
# 系数森林图（按 |coef| 排序，最多显示 15 个变量）
top = coefficients.sort_values("coef", key=abs, ascending=False).head(15)
fig = plot_coefficient_table(top, title="评级变动对宏观变量的回归系数（双向固定效应）")
save_figure(fig, "nb03_coefficient_plot")
fig

## 3. 稳健性检验

* **混合 OLS**（无固定效应）作为对照：系数方向常与固定效应模型一致，但量级不同；
* **按国家×机构聚类**：若同一国家内不同机构的评级行为独立，聚类维度应更细；
* **Logit** 替代线性概率模型：处理二元因变量的函数形式问题。

In [ ]:
robustness: dict[str, object] = {}

robustness["双向FE（主回归）"] = results["rating_change"]
robustness["仅国家FE"] = fit_fixed_effects(
    panel, "rating_change", FEATURES, entity_effects=True, time_effects=False
)
robustness["混合OLS"] = fit_pooled_ols(panel, "rating_change", FEATURES)
robustness["按国家×机构聚类"] = fit_fixed_effects(
    panel, "rating_change", FEATURES, cluster_col="agency", time_effects=True
)

compare_results(robustness).round(4)

In [ ]:
try:
    logit_result = fit_logit(panel, "is_downgrade", FEATURES)
    print(logit_result.summary_text())
except Exception as exc:
    print(f"Logit 未能收敛（常见于完全分离或稀疏事件）：{exc}")

## 4. 分组异质性

同一宏观变量对投资级与投机级国家的影响通常不对称：
高评级国家债务上升未必立刻触发下调，低评级国家则可能迅速失去市场准入。

下面的分组回归仅作演示——**分组会显著减少样本量**，请谨慎解读显著性。

In [ ]:
panel_split = panel.dropna(subset=["score_in_effect"]).copy()
panel_split["投资级"] = np.where(panel_split["score_in_effect"] >= 12, "投资级", "投机级")

for group_name, subset in panel_split.groupby("投资级"):
    if len(subset) < 100:
        print(f"{group_name}: 样本量仅 {len(subset)}，跳过")
        continue
    result = fit_fixed_effects(
        subset, "is_downgrade", FEATURES, time_effects=True
    )
    print(f"\n===== {group_name}（n={result.nobs}）=====")
    print(result.to_frame().query("p_value < 0.1").round(4))